# 03 - Connect, verify identity, and close

Connection configures the USB-to-UART bridge and queries APT identity. It does not enable
or move the motor. An unsupported model, serial mismatch, or channel count is rejected.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

## Prefer a context manager

`with` closes the USB handle on success or failure. The constructor alone does not
connect. Set `SERIAL` and `CONNECT_TO_DEVICE` above before running this cell.


In [ ]:
if CONNECT_TO_DEVICE:
    with configured_mount() as mount:
        identity = mount.info()
        print(asdict(identity))
else:
    print("Connection skipped; enter SERIAL and set CONNECT_TO_DEVICE = True.")

## Explicit lifecycle when integrating another program

This is an alternative connection example, executed after the first has closed.
Keep `connect()` inside `try` so that even a connection failure reaches `finally`.
Do not leave a live mount in a variable across notebook cells.


In [ ]:
if CONNECT_TO_DEVICE:
    mount = configured_mount(query_timeout=3)
    try:
        mount.connect(timeout=5)
        print("Connected to:", mount.info().serial)
    finally:
        mount.close()
else:
    print("Explicit connection example skipped.")

If connection fails, resolve the reported driver, access, ownership, or identity problem
before retrying. The per-process ownership check cannot exclude another notebook kernel
or another application. Closing releases USB; it does not disable the motor or remove its
holding torque.

Next: [04 - Status and position](04_status_and_position.ipynb).
